In [1]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
kategorie = ['elektronika', 'odzież', 'żywność', 'książki']

def generate_transaction():
    return {
        'tx_id': f'TX{random.randint(1000,9999)}',
        'user_id': f'u{random.randint(1,20):02d}',
        'amount': round(random.uniform(5.0, 5000.0), 2),
        'store': random.choice(sklepy),
        'category': random.choice(kategorie),
        'timestamp': datetime.now().isoformat(),
    }

for i in range(1000):
    tx = generate_transaction()
    producer.send('transactions', value=tx)
    print(f"[{i+1}] {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']}")
    time.sleep(2)

producer.flush()
producer.close()

Writing producer.py


In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate

In [5]:
## kafka wersja zająca 
%%file kafka_raw.py
from pyspark.sql import SparkSession
 
KAFKA_PACKAGE = "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0"
 
spark = (
    SparkSession.builder
    .appName("Lab4-Kafka")
    .config("spark.jars.packages", KAFKA_PACKAGE)
    .getOrCreate()
)
 
spark.sparkContext.setLogLevel("WARN")
 
kafka_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "transactions")
    .load()
)
 
q = (
    kafka_raw.writeStream
    .format("console") 
    .outputMode("append") 
    .option("truncate", False)
    .start()
)
q.awaitTermination()

Overwriting kafka_raw.py


In [8]:
%%file kafka_raw.py
from pyspark.sql import SparkSession
 
spark = (
    SparkSession.builder
    .appName("Lab4-Kafka")
    .getOrCreate()
)
 
spark.sparkContext.setLogLevel("WARN")
 
kafka_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "transactions")
    .load()
)

parsed = kafka_raw.select(col("value").cast("string"))
 
q = (
    kafka_raw.writeStream
    .format("console") 
    .outputMode("append")
    .option("truncate", False)
    .start()
)
q.awaitTermination()

Overwriting kafka_raw.py


In [2]:
import os
import pyspark

# Autodetekcja wersji Sparka → właściwy connector
spark_version = pyspark.__version__
print(f"Wykryta wersja PySpark: {spark_version}")

if spark_version.startswith("4"):
    KAFKA_PACKAGE = "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0-preview2"
else:
    # Spark 3.x
    KAFKA_PACKAGE = "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0"

print(f"Użyty connector:        {KAFKA_PACKAGE}")

# Musi być ustawione PRZED SparkSession.builder
os.environ['PYSPARK_SUBMIT_ARGS'] = f'--packages {KAFKA_PACKAGE} pyspark-shell'

Wykryta wersja PySpark: 4.0.0.dev2
Użyty connector:        org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0-preview2


In [9]:
%%file kafka_raw.py
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json,to_timestamp 
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
spark = (
    SparkSession.builder
    .appName("Lab4-Kafka")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
 
SCHEMA = tx_schema = StructType([
    StructField("tx_id",     StringType()),
    StructField("user_id",   StringType()),
    StructField("amount",    DoubleType()),
    StructField("store",     StringType()),
    StructField("category",  StringType()),
    StructField("timestamp", StringType()),
])
 
kafka_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "transactions")
    .load()
)
parsed = (kafka_raw.select(
    from_json(col("value").cast("string"), SCHEMA).alias("tx"))
    .select("tx.*")
    .withColumn("timestamp", to_timestamp("timestamp"))
         )
q = (
    parsed.writeStream
    .format("console") 
    .outputMode("append")
    .option("truncate", False)
    .start()
)
q.awaitTermination()

Overwriting kafka_raw.py
